# Examen MF1


Utiliza el dataset de vehículos para ayudar a nuestra empresa a automatizar el sistema de tasación de coches. Lee detenidamente la descripción de los datos.

El objetivo es que, al recibir un nuevo vehículo, podamos **predecir el precio** de venta óptimo. Tu jefa te dice que en unas pocas horas quiere un primer prototipo.

Este examen consta de 5 apartados, cada uno con subapartados específicos y puntuación asignada. La puntuación global por apartado es:

1. **Entendimiento de los datos**: 1.5 puntos.
2. **Preparación de los datos**: 3.25 puntos.
3. **Modelado**: 3.25 puntos.
4. **Evaluación final**: 1 punto.
5. **Conclusiones**: 1 punto.

Además, se otorgará hasta **1 punto extra** por aportes extraordinarios adicionales. La nota final se calculará como el mínimo(10, nota obtenida).

Consideraciones:
- Para evaluar los modelos, nuestra empresa ha decidido que el **MAE** es la mejor opción en este caso.
- Las preguntas que requieran una justificación escrita estarán indicadas en **negrita** y con frases como "justifica tu decisión" o "comenta".
- En las respuestas no hace falta escribir mucho texto, solo una justificación lógica y razonable.
- Si se pide una justificación o un comentario y no se proporciona, la pregunta puede recibir 0 puntos, independientemente de que el código sea correcto.
- Si el código y la justificación no son coherentes entre sí, la calificación se verá penalizada.
- Si no se pide una explicación, no es necesario proporcionar una.
- Cometer errores de metodología graves (como "data leakage" o un uso inadecuado del conjunto de test) puede suponer una calificación de 0 en los apartados correspondientes.

In [1]:
# Importa librerias, funciones, objetos...

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pprint import pprint
import seaborn as sns
import time
import joblib
import os
from sklearn.tree import plot_tree

# metrics
from sklearn.metrics import make_scorer, mean_absolute_error, median_absolute_error, root_mean_squared_error
from sklearn.metrics import balanced_accuracy_score, precision_score, recall_score

# cross-validation
from sklearn.model_selection import train_test_split, cross_validate, cross_val_predict, KFold, RandomizedSearchCV, GridSearchCV, StratifiedKFold

# preprocessing
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.preprocessing import MinMaxScaler, RobustScaler, StandardScaler
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# regression models
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, ElasticNet
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from tabpfn import TabPFNRegressor

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from tabpfn import TabPFNClassifier

# Class imbalance
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler, SMOTE
from sklearn.tree import DecisionTreeRegressor, plot_tree



c:\Users\Alumne_mati1\Documents\machine_learning_course_cifo_2026\.venv_py312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 1. Entendimiento de los datos (Data Understanding)

## 1.1. Descripción de los datos

### 1.1.1. Carga el dataset de vehículos (0.05 puntos)

In [2]:
df = pd.read_csv('../data_exam/cars.csv')

df.head()

,Price,Year,Kms,Miles,Fuel,Transmission,Owner,Seller,Drivetrain,Length,Width,Height,Seating
0,5656.0,2017,87150,54152.48265,Petrol,Manual,First,Corporate,FWD,3990,1680,1505,5
1,5040.0,2014,75000,46602.82500,Diesel,Manual,Second,Individual,FWD,3995,1695,1555,5
2,2464.0,2011,67000,41631.85700,Petrol,Manual,First,Individual,FWD,3585,1595,1550,5
3,8948.8,2019,37500,23301.41250,Petrol,Manual,First,Individual,FWD,3995,1745,1510,5
4,21840.0,2018,69000,42874.59900,Diesel,Manual,First,Individual,RWD,4735,1830,1795,7


### 1.1.2. Inspección básica (0.45 puntos)

Haz una inspección básica como consideres más oportuno. Por ejemplo ver el tamaño del dataset, primeras filas, tipos de columnas, etc.

**Alguna observación interesante?**

In [3]:
#Se verifica que el dataset tiene 1703 filas y 13 columnas originalmente.
df.shape

(1703, 13)

In [4]:
#Se verifica que hay 6 columnas de tipo int, dos columnas de tipo float, y 5 columnas de tipo str
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1703 entries, 0 to 1702
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Price         1703 non-null   float64
 1   Year          1703 non-null   int64  
 2   Kms           1703 non-null   int64  
 3   Miles         1703 non-null   float64
 4   Fuel          1703 non-null   str    
 5   Transmission  1703 non-null   str    
 6   Owner         1703 non-null   str    
 7   Seller        1701 non-null   str    
 8   Drivetrain    1703 non-null   str    
 9   Length        1703 non-null   int64  
 10  Width         1703 non-null   int64  
 11  Height        1703 non-null   int64  
 12  Seating       1703 non-null   int64  
dtypes: float64(2), int64(6), str(5)
memory usage: 173.1 KB


## 1.2. Exploración de los datos (EDA)

Realiza una visualización básica de los datos.

### 1.2.1. Distribuciones (0.5 puntos)

**Comenta** si observas alguna cosa relevante. En caso de que no haya nada raro o especial, una respuesta del tipo *"No se aprecia nada relevante, ya que (...)."* es perfectamente válida.

In [5]:
#Lo interesante para mi, es que no hay separacion de filas tipo labeled y leadeboard y ahora grego una columa, que sera elm target
#df['proba'] = df['Price'].median

df['SalePrice'] = 5656

### 1.2.2. Correlación (0.5 puntos)

**Comenta** si observas alguna cosa relevante. En caso de que no haya nada raro o especial, una respuesta del tipo *"No se aprecia nada relevante, ya que (...)."* es perfectamente válida.

In [6]:
df

,Price,Year,Kms,Miles,Fuel,Transmission,Owner,Seller,Drivetrain,Length,Width,Height,Seating,SalePrice
0,5656.0,2017,87150,54152.482650,Petrol,Manual,First,Corporate,FWD,3990,1680,1505,5,5656
1,5040.0,2014,75000,46602.825000,Diesel,Manual,Second,Individual,FWD,3995,1695,1555,5,5656
2,2464.0,2011,67000,41631.857000,Petrol,Manual,First,Individual,FWD,3585,1595,1550,5,5656
3,8948.8,2019,37500,23301.412500,Petrol,Manual,First,Individual,FWD,3995,1745,1510,5,5656
4,21840.0,2018,69000,42874.599000,Diesel,Manual,First,Individual,RWD,4735,1830,1795,7,5656
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1698,2744.0,2014,79000,49088.309000,Petrol,Manual,Second,Individual,FWD,3775,1680,1620,5,5656
1699,9520.0,2016,90300,56109.801300,Diesel,Manual,First,Individual,FWD,4585,1890,1785,7,5656
1700,3080.0,2014,83000,51573.793000,Petrol,Manual,Second,Individual,FWD,3495,1550,1500,5,5656
1701,2688.0,2013,73000,45360.083000,Petrol,Manual,First,Individual,FWD,3795,1680,1427,5,5656


# 2. Preparación de los datos

## 2.1. Limpieza

### 2.1.1. Nulos (0.5 puntos)

Hay valores nulos?

En caso afirmativo, explica la que creas que es la mejor opción para tratarlos (eliminarlos, imputarlos en el feature engineering, etc.).

**Justifica tu decisión.**

In [7]:
df.columns[df.isna().any()]

Index(['Seller'], dtype='str')

### 2.1.2. Outliers (0.5 puntos)

Crees que hay algun outlier sospechoso?

En caso afirmativo, explica la que creas que es la mejor opción para tratarlos (eliminarlos, dejarlos, imputarlos en el feature engineering, etc.).

**Justifica tu decisión.**

In [8]:
#df['Seller'].isna().value_counts()
df[df['Seller'].isna() == True]

,Price,Year,Kms,Miles,Fuel,Transmission,Owner,Seller,Drivetrain,Length,Width,Height,Seating,SalePrice
104,15680.0,2014,143000,88856.053,Diesel,Automatic,Second,NaN,RWD,4705,1840,1850,7,5656
1568,17024.0,2021,14000,8699.194,Petrol,Manual,First,NaN,AWD,3985,1820,1844,4,5656


In [9]:
if df['Seller'].isna().any() == True:
    df['Seller'] = df['Seller'].fillna('Individual')

In [10]:
df['Seller'].isna().value_counts()

Seller
False    1703
Name: count, dtype: int64

### 2.1.3. Duplicados (0.5 puntos)

Hay filas duplicadas?

En caso afirmativo, aplica la que creas que es la mejor opción para tratarlas (eliminarlas, no hacer nada...).

**Justifica tu decisión.**

In [11]:
#df.duplicated().any()
df[df.duplicated() == True]

#Al ver los supuestos duplicados, Tienen distintos valores, en varias columnas, por lo tanto asumo que no son duplicados sino parecidos

,Price,Year,Kms,Miles,Fuel,Transmission,Owner,Seller,Drivetrain,Length,Width,Height,Seating,SalePrice
756,3528.0,2017,15507,9635.600097,Petrol,Manual,First,Individual,FWD,3679,1579,1478,5,5656
1005,8120.0,2019,11874,7378.159254,Petrol,Automatic,First,Individual,FWD,3840,1735,1530,5,5656
1286,7268.8,2020,52236,32457.935556,Petrol,Manual,First,Individual,FWD,3765,1660,1520,5,5656
1539,15568.0,2017,56000,34796.776000,Petrol,Automatic,First,Individual,FWD,4670,1814,1476,5,5656


## 2.2. Feature Engineering

### 2.2.1. Encoding (1.5 puntos)

Define los encodings que consideres más adecuados para las variables categóricas. Por ejemplo One-Hot Encoding y/o Ordingal Encoding.

In [13]:
# 1. Crear la variable X amb les característiques predictores
# (eliminem 'SalePrice' perquè no confongui el model)
X = df.drop(['SalePrice'], axis=1)

# 2. Separar l'objectiu en la variable `y`
y = df['SalePrice']

# Convertimos los strings a 'category' de forma nativa para evitar el error en el HistGradient
for col in X.select_dtypes(include=['object']).columns:
    X[col] = X[col].astype('category')
    

# 3. Divisió Entrenament / Test (20% de dades per a test, equivalent a test_size=0.2)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,  # Mantinguem el 20% de l'exemple de tips
    shuffle=True,
    random_state=42
)

# Comprovació de les mides resultants
print(f"Mida de X_train: {X_train.shape} | Mida de y_train: {y_train.shape}")
print(f"Mida de X_test: {X_test.shape} | Mida de y_test: {y_test.shape}")

Mida de X_train: (1362, 13) | Mida de y_train: (1362,)
Mida de X_test: (341, 13) | Mida de y_test: (341,)


C:\Users\Alumne_mati1\AppData\Local\Temp\ipykernel_8504\407721487.py:9: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in X.select_dtypes(include=['object']).columns:


In [14]:
# 1. Separar los nombres de las columnas según su tipo de dato
# Columnas categóricas (texto)
# Identificación automática de tipos de datos para la Pipeline (para LR y RF)
categorical_features = X_train.select_dtypes(include=['category']).columns.tolist()
numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Transformadores con imputación de nulos integrada para evitar fallos en modelos numéricos
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='passthrough'
)

In [15]:
df.shape

(1703, 14)

### 2.2.2. Otros (0.25 puntos)

Haz cualquier otro procesamiento que consideres oportuno. Por ejemplo crear o eliminar columnas, filas, etc.

**Justifica** de forma simple cada una de tus decisiones.

Si decides no hacer nada en este apartado y das una buena **justificación** al respecto, también puedes obtener la puntuación. 

# 3. Modelado

## 3.1. Diseño del testeo y validación

### 3.1.1. Train - Test split (0.25 puntos)

Divide el dataset en conjuntos de train y de test en las proporciones que creas más adecuadas. Considera también otros hiperparámetros como `shuffle`.

**Justifica tus decisiones.**

No es necesario hacer ningún estudio al respecto, solo escoge una proporción que te parezca adecuada. No hay una única solución correcta, cualquier opción *razonable* contará la puntuación máxima.

In [ ]:
#arriba

### 3.1.2. Cross-validation (0.25 puntos)

Crea el objeto de cross-validation que consideres más oportuno. Solo tienes que crear el objecto para definir el tipo de cross-validation que van a emplear los modelos en el siguiente apartado.

**Justifica las decisiones.**

In [ ]:
#Usamos diez grupos para hacer el cross validation para que sea facil y rapido la validacion de las muestras.
kf = KFold(n_splits=10, shuffle=True, random_state=42)

In [16]:
kf = KFold(n_splits=10, shuffle=True, random_state=42)

## 3.2. Modelos de ML

Pureba diferentes modelos de ML con sus correspondientes hiperparámetros. Asegurate de que estás empleando cross-validation en el conjunto de train en todo momento.

- Para cada uno de los modelos, muestra su rendimiento en el conjunto de train y de validación (llamado *test* en muchas de las funciones de `sklearn`).
- Emplea las técnicas que consideres más oportunas, por ejemplo búsqueda de los mejores hiperparámetros, etc.
- No es necesario probar tropecientos modelos, puedes usar tu intuición y conocimiento para probar solo 2 o 3 que creas que van a tener el mejor rendimiento.
- Procura mantener esta sección un poco limpia.

### 3.2.1. Baseline (0.25 puntos)

Utilitza un modelo baseline (dummy) para compararlo con los siguientes modelos (más avanzados) y comprobar que, efectivamente, son mejores que este.

**Justifica tu elección de modelo "baseline".**

In [17]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.dummy import DummyRegressor

dummy_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('dummy', DummyRegressor(strategy='mean'))
])

dummy_cv = cross_validate(dummy_pipeline, X_train, y_train, cv=kf, scoring='neg_mean_absolute_error', return_train_score=True, n_jobs=-1)

print('CV Train MAE:', -dummy_cv['train_score'].mean().round(2))
print('CV Validation MAE:', -dummy_cv['test_score'].mean().round(2))

CV Train MAE: -0.0
CV Validation MAE: -0.0


### 3.2.2. Entrenamiento y validación de modelos (2 puntos)

Elige entre 2 y 3 de los modelos trabajados en clase, aquellos que esperas que tengan potencial de dar mejor rendimiento.

**Justifica porqué pruebas estos modelos y no otros.**

Entrénalos y valídalos (usando cross-validation), realizando la búsqueda de hiperparámetros (hyperparameter tuning) si es necesario, con el método que consideres más adecuado.

In [ ]:
#Regresión Lineal

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

lr_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('lr', LinearRegression())
])

lr_cv = cross_validate(lr_pipeline, X_train, y_train, cv=kf, scoring='neg_mean_absolute_error', return_train_score=True, n_jobs=-1)

print('CV Train MAE:', -lr_cv['train_score'].mean().round(2))
print('CV Validation MAE:', -lr_cv['test_score'].mean().round(2))

In [ ]:
#random forest
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('rf', RandomForestRegressor(random_state=42))
])

rf_cv = cross_validate(rf_pipeline, X_train, y_train, cv=kf, scoring='neg_mean_absolute_error', return_train_score=True, n_jobs=-1)

print('CV Train MAE:', -rf_cv['train_score'].mean().round(2))
print('CV Validation MAE:', -rf_cv['test_score'].mean().round(2))

In [18]:
# HistGradient Boosting (Tuned)
from sklearn.ensemble import HistGradientBoostingRegressor
gb = HistGradientBoostingRegressor(categorical_features='from_dtype', random_state=42)
gb_param_dist = {
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_iter': [50, 100, 150, 200],
    'max_depth': range(5, 51),
    'min_samples_leaf': range(10, 31),
    'l2_regularization': [0.0, 0.1, 1.0, 10.0]
}

gb_rs = RandomizedSearchCV(
    estimator=gb,
    param_distributions=gb_param_dist,
    n_iter=50,
    scoring='neg_mean_absolute_error',
    return_train_score=True,
    cv=kf,
    random_state=42,
    n_jobs=-1
)
gb_rs.fit(X_train, y_train)

print('Best RandomizedSearchCV parameters:', gb_rs.best_params_)
print('CV Train MAE:', -gb_rs.cv_results_['mean_train_score'][gb_rs.best_index_].round(2))
print('CV Validation MAE:', -gb_rs.cv_results_['mean_test_score'][gb_rs.best_index_].round(2))

Best RandomizedSearchCV parameters: {'min_samples_leaf': 26, 'max_iter': 150, 'max_depth': 32, 'learning_rate': 0.1, 'l2_regularization': 10.0}
CV Train MAE: -0.0
CV Validation MAE: -0.0


### 3.2.3. Conclusiones del entrenamiento (0.5 puntos)

Qué modelos han dado un mejor rendimiento? Sabes o intuyes porqué?

In [ ]:
#Me ha dado mejor resultado el modelo de Tabpfn pero por alguana razon no me permite usar mas de 1000 filas, 
# intuyo porque ya tiene preentranado muchos parametros 
# que podrian conicidior con las features utilizades en este trabajo, por otro lado he usado el modelo HistGradient Boosting (Tuned), que se ha comportado mejor



# 4. Evaluación final de modelos

### 4.1. Test (0.5 puntos)

Escoge algunos de los mejores modelos que has encontrado en el apartado anterior:
1. Entrénalos con todo el conjunto de train (solo si aún no los tienes entrenados con todo este conjunto).
2. Una vez entrenados, comprueba su rendimiento final en el conjunto de test.

En este apartado solo es necesario que haya código, en el apartado siguiente ya escribirás tus conclusiones.

In [21]:
# 1. Filtramos las filas del leaderboard conservando el 'Id' original
df_testing = X_test.copy()

# 2. Creamos la matriz de características X_leaderboard
X_testing = df_testing.drop(columns=['SalePrice'], errors='ignore')

# 4. El modelo ganador realiza las predicciones
predicciones_testing = gb_rs.predict(X_testing)

# 5. Estructuramos el DataFrame final con el nombre exigido: 'prediction'
submission_df = pd.DataFrame({
    'SalePrice': predicciones_testing.round(2)  
})

# 6. Exportamos el nuevo archivo CSV corregido
submission_df.to_csv('cars_SalePrice.csv', index=False)

print("--- ARCHIVO GENERADO CON ÉXITO ---")
print(submission_df.head())

--- ARCHIVO GENERADO CON ÉXITO ---
   SalePrice
0     5656.0
1     5656.0
2     5656.0
3     5656.0
4     5656.0


### 4.2. Conclusiones del test (0.5 puntos)

Qué modelos han dado un mejor rendimiento? Sabes o intuyes porqué?

In [ ]:
#al hacer el test se observa que se genera solamente la columna de precios de venta 'SalePrice'

resultados = pd.DataFrame({
    'Model': [
        'Dummy',
        'Linear Regression',
        'Random Forest (Base)',
        'HistGradient Boosting (Tuned)'
    ],
    'Validation MAE': [
        -dummy_cv['test_score'].mean().round(2),
        -lr_cv['test_score'].mean().round(2),
        -rf_cv['test_score'].mean().round(2),
        -gb_rs.cv_results_['mean_test_score'][gb_rs.best_index_].round(2)
    ]
})

print(resultados.to_string(index=False))

# 5. Conclusiones (1 punto)

**Escribe** tus conclusiones finales.
- Con qué modelo te quedarías para poner en producción? En caso de que no haya ninguno explica porqué.
- Si tuvieras que mejorar el rendimiento del modelo, cuáles son los siguientes pasos que seguirías?

In [ ]:
#La estimacion del nuevo precio de venta no se imcrementara en mnuchos casos